In [2]:
import csv
import pandas as pd
from pathlib import Path


def _read_evolving_csv(csv_path: Path) -> pd.DataFrame:
    """
    Liest experiment_summary.csv auch dann fehlerfrei ein, wenn neuere Runs
    zusätzliche Spalten (z.B. mean_c_at_best_loss, mean_c_last_epoch) haben als ältere Runs.
    """
    try:
        return pd.read_csv(csv_path)
    except pd.errors.ParserError:
        with open(csv_path, "r") as f:
            rows = list(csv.reader(f))
        header = rows[0]
        max_cols = max(len(r) for r in rows)
        idx = header.index("fold_c_indices") + 1
        if max_cols > len(header):
            extra_cols = ["mean_c_at_best_loss", "std_c_at_best_loss", "mean_c_last_epoch", "std_c_last_epoch"]
            header = header[:idx] + extra_cols[:max_cols - len(header)] + header[idx:]

        padded_rows = [
            r[:idx] + [None] * (max_cols - len(r)) + r[idx:] if len(r) < max_cols else r
            for r in rows[1:]
        ]
        df = pd.DataFrame(padded_rows, columns=header)
        for col in df.columns:
            if col.startswith(("mean_", "std_", "lr", "dropout", "loss_alpha", "weight_decay")):
                df[col] = pd.to_numeric(df[col], errors="coerce")
        return df


# Ordner (relativ zu diesem Notebook) -> Anzeigename in der Tabelle
MODEL_DIRS = {
    "ABMIL":    "ABMIL (wsi only)",
    "TRANSMIL": "TransMIL(wsi only)",
    "MCAT":     "MCAT (wsi + genomic)",
    "MOTCAT":   "MOTCAT (wsi + genomic)",
    "SURVPATH": "SurvPath (wsi + genomic)",
    "SNN":      "SNN (genomic only)",
    "MLP":      "MLP (genomic only)",
}
DATASETS = ["KIRC", "GBMLGG", "KIRP"]


def create_last_epoch_table(
    base_dir: Path = Path(".."),
    model_dirs: dict = MODEL_DIRS,
    datasets: list = DATASETS,
    decimals: int = 3,
) -> pd.DataFrame:
    """
    Fasst pro Modell-Ordner den jeweils neuesten Run je Kohorte zusammen und zeigt
    nur den C-Index @ Last Epoch (mean ± std über die Folds) pro Dataset sowie Overall.
    Overall = Mittel der Dataset-Means ± Mittel der Dataset-Stds (nur wenn alle Datasets vorliegen).
    """
    fmt = lambda m, s: f"{m:.{decimals}f} ± {s:.{decimals}f}"
    rows = []
    for folder, name in model_dirs.items():
        row = {"Model": name, **{d: "–" for d in datasets}, "Overall": "–"}
        csv_path = Path(base_dir) / folder / "runs" / "experiment_summary.csv"
        if not csv_path.exists():
            rows.append(row)
            continue

        df = _read_evolving_csv(csv_path)
        # Neuere Runs (SNN, MLP, SurvPath) speichern nur mean_c_index, waehlen aber immer die letzte Epoche
        # (model_selection == "last_epoch") -> dann ist mean_c_index der C-Index @ Last Epoch.
        if "model_selection" in df.columns:
            last = df["model_selection"] == "last_epoch"
            for stat in ("mean", "std"):
                col = f"{stat}_c_last_epoch"
                if col not in df.columns:
                    df[col] = float("nan")
                df.loc[last, col] = df.loc[last, col].fillna(df.loc[last, f"{stat}_c_index"])
        if "mean_c_last_epoch" not in df.columns:
            rows.append(row)
            continue

        # Neuesten Run pro Kohorte anhand des Zeitstempels im run_name (..._YYYYMMDD_HHMMSS)
        df["_ts"] = df["run_name"].str.extract(r"(\d{8}_\d{6})$")[0]
        df = df.sort_values("_ts").drop_duplicates(subset="cohort", keep="last")
        df["Dataset"] = df["cohort"].str.replace("TCGA-", "", regex=False)
        df = df.dropna(subset=["mean_c_last_epoch"]).set_index("Dataset")

        means, stds = [], []
        for d in datasets:
            if d in df.index:
                m, s = float(df.loc[d, "mean_c_last_epoch"]), float(df.loc[d, "std_c_last_epoch"])
                row[d] = fmt(m, s)
                means.append(m)
                stds.append(s)
        if len(means) == len(datasets):
            row["Overall"] = fmt(sum(means) / len(means), sum(stds) / len(stds))
        rows.append(row)

    return pd.DataFrame(rows)


summary_table = create_last_epoch_table()
display(summary_table.style.hide(axis="index"))


Model,KIRC,GBMLGG,KIRP,Overall
ABMIL (wsi only),0.697 ± 0.088,0.750 ± 0.071,0.757 ± 0.105,0.735 ± 0.088
TransMIL(wsi only),0.708 ± 0.062,0.755 ± 0.067,0.749 ± 0.096,0.737 ± 0.075
MCAT (wsi + genomic),0.734 ± 0.046,0.787 ± 0.036,0.818 ± 0.120,0.779 ± 0.067
MOTCAT (wsi + genomic),0.725 ± 0.034,0.788 ± 0.039,0.818 ± 0.135,0.777 ± 0.069
SurvPath (wsi + genomic),0.718 ± 0.048,0.826 ± 0.049,0.789 ± 0.127,0.777 ± 0.075
SNN (genomic only),0.702 ± 0.063,0.818 ± 0.033,0.820 ± 0.124,0.780 ± 0.073
MLP (genomic only),0.705 ± 0.054,0.830 ± 0.043,0.791 ± 0.141,0.775 ± 0.079
